<!--    
    ================================================================================================
    
    Copyright (c) 2026 Ilya Snegov

    Licensed under the Apache License, Version 2.0 (the "License");
    you may not use this file except in compliance with the License.
    You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

    ================================================================================================
    notebooks/02-feature-engineering.ipynb
    ================================================================================================
-->

# **Feature Engineering**

*A Jupyter notebook for generating technical indicators using TA-Lib, creating temporal features, and performing feature selection from preprocessed cryptocurrency OHLCV data.*

## **I. Setup the environment**

### **Import required libraries**

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

from finml_lib import set_matplotlib_style
from finml_lib.features import (
    create_moving_averages_indicator_group,
    create_overlap_studies_indicator_group,
    create_momentum_index_indicator_group,
    create_oscillator_indicator_group,
    create_momentum_indicators_indicator_group,
    create_volume_cycle_indicator_group,
    create_price_volatility_indicator_group,
    create_pattern_recognition_indicator_group,
    create_time_recognition_indicator_group,
    analyze_nan_distribution,
    remove_zero_variance_features,
    select_features_by_correlation_clustering
)

### **Apply global Matplotlib style for consistent visualization**

In [ ]:
set_matplotlib_style()

### **Load the train dataset**

In [ ]:
train_df = pd.read_parquet("./data/train.parquet")

### **Define configuration parameters and extract OHLCV arrays for indicator generation**

In [ ]:
# Lookback periods for technical indicator generation
SHORT = 24      # 1 day (24 hours); determines the lookback window for short-term indicators
MEDIUM = 72     # 3 days (72 hours); determines the lookback window for medium-term indicators
LONG = 168      # 1 week (168 hours); determines the lookback window for long-term indicators

SEED = 42       # Global random seed ensuring reproducibility when constructing the baseline 
                # feature importance map

CORR_THRESHOLD = 0.85   # Absolute correlation threshold for clustering and selecting representative features

# Pre-extract OHLCV data as float64 numpy arrays to eliminate redundant type conversions and indexing 
# overhead during subsequent indicator generation
open_prices = train_df["open"].to_numpy(dtype=np.float64)       # Open price array for indicator computation
high_prices = train_df["high"].to_numpy(dtype=np.float64)       # High price array for indicator computation
low_prices = train_df["low"].to_numpy(dtype=np.float64)         # Low price array for indicator computation
close_prices = train_df["close"].to_numpy(dtype=np.float64)     # Close price array for indicator computation
volume = train_df["volume"].to_numpy(dtype=np.float64)          # Volume array for indicator computation

## **II. Construct all technical and time-based indicator features**

In [ ]:
train_df = create_moving_averages_indicator_group(train_df, close_prices, SHORT, MEDIUM, LONG)
train_df = create_overlap_studies_indicator_group(train_df, close_prices, high_prices, low_prices, SHORT, MEDIUM, LONG)
train_df = create_momentum_index_indicator_group(train_df, high_prices, low_prices, close_prices, volume, SHORT, MEDIUM, LONG)
train_df = create_oscillator_indicator_group(train_df, open_prices, high_prices, low_prices, close_prices, SHORT, MEDIUM, LONG)
train_df = create_momentum_indicators_indicator_group(train_df, high_prices, low_prices, close_prices, SHORT, MEDIUM, LONG)
train_df = create_volume_cycle_indicator_group(train_df, high_prices, low_prices, close_prices, volume)
train_df = create_price_volatility_indicator_group(train_df, open_prices, high_prices, low_prices, close_prices, SHORT, MEDIUM, LONG)
train_df = create_pattern_recognition_indicator_group(train_df, open_prices, high_prices, low_prices, close_prices)
train_df = create_time_recognition_indicator_group(train_df)
train_df

## **III. Inspect the missing value distribution**

In [ ]:
analyze_nan_distribution(train_df)

## **IV. Analysis of missing value distribution in engineered features**

### **Overview**

The feature engineering process generated 502 missing values, concentrated in a single continuous block at the initial timestamp (2019-09-08 to 2019-09-29), representing approximately $1.9\%$ of the total dataset. These missing values are an inherent artifact of the TA-Lib technical indicators, which require a minimum lookback period (warmup) to compute moving averages, oscillators, and volatility metrics. Without sufficient historical context, the initial calculations remain mathematically undefined.

### **Evaluation of handling strategies**

| Strategy | Mechanism |
| :--- | :--- |
| **Forward/Backward Fill** | Propagates the last known value forward or the next known value backward to fill gaps. |
| **Interpolation (Linear/Spline)** | Estimates missing values by mathematically bridging surrounding data points. |
| **Random Row Deletion** | Removes rows containing missing values randomly across the dataset. |
| **Dropping the Initial Block** | Removes the contiguous block of undefined rows at the beginning of the series. |

### **Conclusion**

In the context of time series modeling, handling missing values requires strict adherence to temporal causality. **Forward/Backward Fill** and **Interpolation** (Linear/Spline) are fundamentally flawed for this context. Both methods synthesize artificial market dynamics — effectively "inventing" price behavior — rather than reflecting true trading activity. This introduces synthetic artifacts that distort the initial volatility regime and can mislead the model during the learning phase.

Therefore, **Row Deletion** is the optimal strategy. While random row deletion is generally unsuitable for time series as it destroys temporal continuity, it is logically sound in this specific scenario because the missingness is strictly confined to a single contiguous block at the beginning of the series. Dropping this initial undefined block preserves the sequential structure of the remaining data. Since it constitutes a marginal fraction ($1.9\%$) of the total observations, this approach ensures the model is trained exclusively on authentic, observed market data, maintaining the statistical integrity of the feature space without introducing fabricated values.

## **V. Remove missing values**

In [ ]:
train_df = train_df.dropna()

## **VI. Remove zero-variance features**

In [ ]:
train_df = remove_zero_variance_features(train_df)

## **VII. Select the most important predictive features via baseline model evaluation**

### **Extract the feature matrix and target vector**

In [ ]:
X_train = train_df.drop(columns=['target'])
y_train = train_df['target']

### **Train a baseline XGBRegressor**

In [ ]:
baseline_model = XGBRegressor(
    random_state=SEED,
    n_jobs=1
)

baseline_model.fit(X_train, y_train)

### **Extract and rank feature importances**

In [ ]:
feature_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': baseline_model.feature_importances_
}).sort_values(by='importance', ascending=False)

feature_importance_df

### **Filter out zero-importance feature names and convert the result to a Series**

In [ ]:
feature_importances = (
    feature_importance_df[feature_importance_df['importance'] > 0]
    .set_index('feature')['importance']
)

feature_importances

### **Filter the feature matrix to retain non-zero importance features**

In [ ]:
X_train_filtered = X_train[feature_importances.index]

### **Select representative features via correlation clustering**

In [ ]:
selected_features = select_features_by_correlation_clustering(
    X=X_train_filtered,
    feature_importances=feature_importances,
    threshold=CORR_THRESHOLD
)

print(f"Selected {len(selected_features)} features out of {X_train.shape[1]}.")
print(selected_features)

## **VIII. Construct a reusable feature extraction pipeline**

Manually generating hundreds of technical indicators and subsequently filtering them is computationally inefficient and introduces redundancy, particularly during the inference phase or when deploying the model. To streamline the preprocessing stage, the entire sequence — indicator generation, missing value removal, zero-variance filtering, and correlation-based selection — is encapsulated into a single deterministic function.

This function accepts a raw OHLCV DataFrame with a datetime index and outputs strictly the curated feature matrix, serving as the primary preprocessing step before model ingestion. The implementation is defined in the project library at [finml_features.create_features](../packages/finml-features/src/finml_features/__init__.py).


In [ ]:
from finml_features import create_features

## **IX. Exclude feature scaling from the preprocessing pipeline**

Feature scaling is intentionally omitted from the data preprocessing pipeline. The predictive model relies on Gradient Boosting Decision Trees (XGBoost), which partition the feature space using threshold-based splits. Because scaling applies a monotonic transformation, it preserves the relative ordering of feature values, leaving the optimal split points mathematically invariant. Consequently, standardization or normalization provides no improvement to the model's predictive performance and is excluded to maintain computational efficiency and the direct interpretability of the feature space.

## **X. Apply the identical preprocessing pipeline to all datasets**

### **Preprocess the train dataset**

In [ ]:
train_df = pd.read_parquet("./data/train.parquet")

X_train = train_df.drop(columns=['target'])
y_train = train_df['target']

featured_X_train = create_features(X_train)

preprocessed_train_df = featured_X_train.join(y_train)
preprocessed_train_df.to_parquet("./data/preprocessed_train.parquet")

### **Preprocess the validation dataset**

In [ ]:
valid_df = pd.read_parquet("./data/valid.parquet")

X_valid = valid_df.drop(columns=['target'])
y_valid = valid_df['target']

featured_X_valid = create_features(X_valid)

preprocessed_valid_df = featured_X_valid.join(y_valid)
preprocessed_valid_df.to_parquet("./data/preprocessed_valid.parquet")

### **Preprocess the test dataset**

In [ ]:
test_df = pd.read_parquet("./data/test.parquet")

X_test = test_df.drop(columns=['target'])
y_test = test_df['target']

featured_X_test = create_features(X_test)

preprocessed_test_df = featured_X_test.join(y_test)
preprocessed_test_df.to_parquet("./data/preprocessed_test.parquet")